<a href="https://colab.research.google.com/github/Aaron-Bensier/custom-object-detection-yolov5/blob/main/Copy_of_Anomlay_Detection_Avenue_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tracking + Anomaly Detection in Avenue Dataset

## Dataset used: **Avenue Dataset for Abnormal Event Detection**



This dataset accompanies paper "Abnormal Event Detection at 150 FPS in Matlab, Cewu Lu, Jianping Shi, Jiaya Jia, International Conference on Computer Vision, (ICCV), 2013"









`NOTE:` This is just a reference code. It is not mandatory to use the same code, make changes as u need

### Importing required libraries

In [ ]:
import os
import cv2
import torch
import shutil
import random
import numpy as np
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

np.random.seed(42)

### Mount google drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

```
Avenue Dataset
 └── testing_videos
 └── testing_vol
 └── training_videos
 └── training_vol
     
```

In [ ]:
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5.git

Cloning into 'yolov5'...
remote: Enumerating objects: 17516, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 17516 (delta 6), reused 0 (delta 0), pack-reused 17497 (from 4)
Receiving objects: 100% (17516/17516), 16.62 MiB | 12.98 MiB/s, done.
Resolving deltas: 100% (12001/12001), done.


In [ ]:
%cd yolov5/
!pwd

/content/yolov5
/content/yolov5


In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
%cd /content

/content


In [ ]:
# Clone YOLOv5 repository
!git clone https://github.com/ultralytics/yolov5.git



Cloning into 'yolov5'...
remote: Enumerating objects: 17516, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 17516 (delta 6), reused 0 (delta 0), pack-reused 17497 (from 4)
Receiving objects: 100% (17516/17516), 16.62 MiB | 17.40 MiB/s, done.
Resolving deltas: 100% (12001/12001), done.


#### Use the best YOLOv5 model u have trained

In [ ]:
model_path ='/content/drive/MyDrive/best.pt'

In [ ]:
# Load YOLOv5 model
model = torch.hub.load('yolov5', 'custom', path= model_path, source='local')
model.conf = 0.25
model.iou = 0.45

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


YOLOv5 🚀 v7.0-422-g2540fd4c Python-3.11.13 torch-2.6.0+cu124 CPU

Fusing layers... 
Model summary: 212 layers, 20869098 parameters, 0 gradients, 47.9 GFLOPs
Adding AutoShape... 


In [ ]:
! pip install deep_sort_realtime opencv-python tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 49.0 MB/s eta 0:00:00


In [ ]:
from deep_sort_realtime.deepsort_tracker import DeepSort

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
output_video_folder ='/content/drive/MyDrive/Avenue_Anomaly_Results/Processed_Videos'
os.makedirs(output_video_folder, exist_ok = True)

In [ ]:
input_video_path  = '/content/drive/MyDrive/Avenue_Dataset_Extracted/Avenue Dataset/testing_videos/'
output_video_path = '/content/drive/MyDrive/Avenue_Anomaly_Results/Processed_Videos/processed_04.avi'



### Tune the below parameters

In [ ]:
N_history = 5           # how many frames to use for velocity calc
smoothing_frames = 10     # Smoothing: once RED, stay RED for N frames
velocity_threshold = 100   # Velocity threshold (pixels per second)

In [ ]:
import os
import glob
import cv2
import torch
import numpy as np
from tqdm.notebook import tqdm
import warnings

# --- 0. Initial Setup (Run these cells first if starting a new Colab session) ---

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully.")

# Change current directory to /content/ for cloning
%cd /content/

# Clone YOLOv5 repository if it doesn't exist. This is crucial for torch.hub.load.
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5.git
    print("YOLOv5 repository cloned.")
else:
    print("YOLOv5 repository already exists.")

# Install YOLOv5 requirements. Always good to run on a new session.
%pip install -r yolov5/requirements.txt --quiet
print("YOLOv5 requirements installed.")

# Install deep_sort_realtime if not already installed.
try:
    from deep_sort_realtime.deepsort_tracker import DeepSort
    print("deep_sort_realtime already installed.")
except ImportError:
    print("deep_sort_realtime not found. Installing...")
    !pip install deep_sort_realtime
    from deep_sort_realtime.deepsort_tracker import DeepSort
    print("deep_sort_realtime installed.")

# Filter warnings (e.g., FutureWarnings from libraries for cleaner output)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 1. Define Paths and Parameters ---

# Output folder for processed videos in Google Drive
output_video_folder = '/content/drive/MyDrive/Avenue_Anomaly_Results/Processed_Videos'
os.makedirs(output_video_folder, exist_ok=True)
print(f"Output videos will be saved to: {output_video_folder}")

# Input folder containing all test videos from the Avenue Dataset
# Ensure this path is correct based on where you unzipped the dataset
input_video_folder = '/content/drive/MyDrive/Avenue_Dataset_Extracted/Avenue Dataset/testing_videos/'
print(f"Input videos will be read from: {input_video_folder}")

# Anomaly detection parameters (these are the parameters from your image)
N_history = 5            # How many frames to use for velocity calculation
smoothing_frames = 10    # Once an anomaly is detected, keep highlighting for this many frames
velocity_threshold = 100 # Velocity threshold for anomaly (pixels per second)
print(f"Anomaly parameters: N_history={N_history}, smoothing_frames={smoothing_frames}, velocity_threshold={velocity_threshold}")

# --- 2. Load YOLOv5 Model ---

# Ensure 'best.pt' (your trained YOLOv5 model weights) is in your Google Drive's root
# or specify its correct path.
model_path = '/content/drive/MyDrive/best.pt'

try:
    # Load YOLOv5 model using the local cloned repository
    # 'yolov5' refers to the folder name that was cloned, where hubconf.py resides.
    model = torch.hub.load('yolov5', 'custom', path=model_path, source='local')
    model.conf = 0.25 # NMS confidence threshold (adjust if needed)
    model.iou = 0.45  # NMS IoU threshold (adjust if needed)
    print("YOLOv5 model loaded successfully.")
except Exception as e:
    print(f"ERROR: Could not load YOLOv5 model: {e}")
    print("Please ensure 'yolov5' repository is cloned and 'best.pt' path is correct.")
    # Exit or raise error if model loading fails
    raise SystemExit("Exiting due to model loading error.")

# --- 3. Initialize DeepSORT Tracker ---
# Tracker is re-initialized for each video inside the loop to reset its state.

# --- 4. Process Videos in a Loop (Starting from 05.avi) ---

# Get a sorted list of all .avi files in the input folder
all_video_files = glob.glob(os.path.join(input_video_folder, '*.avi'))
all_video_files.sort() # Sort to ensure consistent numerical order (01.avi, 02.avi, ..., 21.avi)

# Filter the list to process videos starting from '05.avi'
video_files_to_process = [f for f in all_video_files if os.path.basename(f) >= '05.avi']

print(f"\nFound {len(video_files_to_process)} video files to process (starting from 05.avi):")
for f_path in video_files_to_process:
    print(f"- {os.path.basename(f_path)}")

print("\nStarting batch processing of videos...")

# Loop through each video file selected for processing
# --- This assumes all setup (imports, model loading, DeepSORT init, paths, parameters) is done above ---

# Loop through each video file selected for processing
for current_input_video_path in video_files_to_process:
    video_filename = os.path.basename(current_input_video_path)
    output_video_filename = f"processed_{video_filename}"
    current_output_video_path = os.path.join(output_video_folder, output_video_filename)

    # Initialize video capture for the current video
    cap = cv2.VideoCapture(current_input_video_path)
    if not cap.isOpened():
        print(f"ERROR: Could not open video file: {current_input_video_path}. Skipping.")
        continue

    # Get video information
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Prepare output video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for output video
    out = cv2.VideoWriter(current_output_video_path, fourcc, fps, (width, height))
    if not out.isOpened():
        print(f"ERROR: Could not open video writer for: {current_output_video_path}. Skipping.")
        cap.release()
        continue

    # Re-initialize DeepSORT tracker and memory for each new video
    tracker = DeepSort(max_age=30)
    track_memory = {}
    anomaly_memory = {}
    saved_anomaly_ids = set()

    read_count = 0
    frame_idx = 0
    pbar = tqdm(total=frame_count, desc=f"Processing {video_filename}") # Progress bar for current video

    # Main video processing loop (frame by frame)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        annotated_frame = frame.copy() # Create a copy of the frame to draw on

        # YOLO Inference: Detect objects
        results = model(frame)
        detections = results.xyxy[0].cpu().numpy()

        # Format detections for DeepSORT
        formatted_detections = []
        for *xyxy, conf, cls in detections:
            xmin, ymin, xmax, ymax = map(int, xyxy)
            box_xywh = [xmin, ymin, xmax - xmin, ymax - ymin]
            formatted_detections.append([box_xywh, conf, int(cls)])

        # Update DeepSORT tracker
        tracks = tracker.update_tracks(formatted_detections, frame=frame)

        # Process each tracked object and apply anomaly logic
        for track in tracks:
            if not track.is_confirmed() or track.time_since_update > 1:
                continue

            track_id = track.track_id
            ltrb = track.to_ltrb()
            xmin, ymin, xmax, ymax = map(int, ltrb)
            center_x, center_y = (xmin + xmax) / 2, (ymin + ymax) / 2

            # Update history for velocity calculation
            track_memory.setdefault(track_id, []).append((frame_idx, center_x, center_y))
            track_memory[track_id] = track_memory[track_id][-N_history:]

            # --- Anomaly Logic (Speed) ---
            current_frame_is_anomaly = False
            anomaly_reason = ""
            if len(track_memory[track_id]) == N_history:
                f1, x1, y1 = track_memory[track_id][0]
                f2, x2, y2 = track_memory[track_id][-1]
                dt = (f2 - f1) / fps if fps > 0 else 1e-6
                pixel_distance = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
                current_velocity = pixel_distance / dt
                if current_velocity > velocity_threshold:
                    current_frame_is_anomaly = True
                    anomaly_reason = "High Speed"
            # Add Direction Change and Forbidden Regions logic here if you implement them

            # Apply Anomaly Smoothing
            is_display_anomaly = False
            if current_frame_is_anomaly:
                anomaly_memory[track_id] = frame_idx
            if track_id in anomaly_memory:
                if frame_idx - anomaly_memory[track_id] <= smoothing_frames:
                    is_display_anomaly = True
                    if not anomaly_reason: anomaly_reason = "Anomaly"
                else:
                    del anomaly_memory[track_id]

            # --- Drawing on annotated_frame ---
            label = f"ID: {track_id}"
            color = (0, 255, 0) # Green for normal
            thickness = 1
            font_scale = 0.5

            if is_display_anomaly:
                label += f" ({anomaly_reason})"
                color = (0, 0, 255) # Red for anomaly
                thickness = 2
                font_scale = 0.6
                if track_id not in saved_anomaly_ids: # Log anomaly to console
                    print(f"  ALERT: Track {track_id} - {anomaly_reason} at frame {frame_idx} of {video_filename}")
                    saved_anomaly_ids.add(track_id)

            cv2.rectangle(annotated_frame, (xmin, ymin), (xmax, ymax), color, thickness)
            cv2.putText(annotated_frame, label, (xmin, ymin - 10), cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, thickness)

        # Write the processed (annotated) frame to the output video file
        out.write(annotated_frame)

        read_count += 1
        frame_idx += 1
        pbar.update(1) # Update the progress bar

    # Release resources for the current video after processing all its frames
    pbar.close()
    cap.release()
    out.release()

    # Final print for current video (can be removed if you truly want zero output besides progress bar)
    print(f"Finished processing {video_filename}. Frames read: {read_count} / {frame_count}. Output saved to: {current_output_video_path}")

print("\n--- All selected videos processed! ---")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.
/content
YOLOv5 repository already exists.


YOLOv5 🚀 v7.0-422-g2540fd4c Python-3.11.13 torch-2.6.0+cu124 CPU



YOLOv5 requirements installed.
deep_sort_realtime already installed.
Output videos will be saved to: /content/drive/MyDrive/Avenue_Anomaly_Results/Processed_Videos
Input videos will be read from: /content/drive/MyDrive/Avenue_Dataset_Extracted/Avenue Dataset/testing_videos/
Anomaly parameters: N_history=5, smoothing_frames=10, velocity_threshold=100


Fusing layers... 
Model summary: 212 layers, 20869098 parameters, 0 gradients, 47.9 GFLOPs
Adding AutoShape... 


YOLOv5 model loaded successfully.

Found 17 video files to process (starting from 05.avi):
- 05.avi
- 06.avi
- 07.avi
- 08.avi
- 09.avi
- 10.avi
- 11.avi
- 12.avi
- 13.avi
- 14.avi
- 15.avi
- 16.avi
- 17.avi
- 18.avi
- 19.avi
- 20.avi
- 21.avi

Starting batch processing of videos...


...

Finished processing 19.avi. Frames read: 248 / 248. Output saved to: /content/drive/MyDrive/Avenue_Anomaly_Results/Processed_Videos/processed_19.avi


Processing 20.avi:   0%|          | 0/273 [00:00<?, ?it/s]

Finished processing 20.avi. Frames read: 273 / 273. Output saved to: /content/drive/MyDrive/Avenue_Anomaly_Results/Processed_Videos/processed_20.avi


Processing 21.avi:   0%|          | 0/76 [00:00<?, ?it/s]

Finished processing 21.avi. Frames read: 76 / 76. Output saved to: /content/drive/MyDrive/Avenue_Anomaly_Results/Processed_Videos/processed_21.avi

--- All selected videos processed! ---




####  Check for different test videos